## Model Monitoring Pipeline

**Steps to be followed**
- Load the Validation data as Reference Data
- Inference data as current data
- Load model for Prediction purpose

In [1]:
import pandas as pd
import pickle
import os
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.test_suite import TestSuite
from evidently.test_preset import DataDriftTestPreset, DataQualityTestPreset

### Load Validation Data as Reference Data

In [2]:
current_dir = os.getcwd()
file_path = os.path.join(current_dir, "..", "data","shap_data", "Validation_data.csv")
file_path = os.path.abspath(file_path)

df_reference = pd.read_csv(file_path)
print(df_reference.shape)
print('')
df_reference.head()

(900, 33)



,Unnamed: 0,NPI_ID,HCP_ID,Age,Number_of_Rx,Rx_last_1_Month,Rx_last_3_Month,Rx_last_6_Month,Rx_last_12_Month,Claims_last_1_Month,...,Promotional_medscape_last_6_month,F2F_visit,F2F_visit_last_1_month,F2F_visit_last_3_month,VRC_visit_last_3_month,VRC_visit_last_6_month,VRC_visit_last_12_month,HCO_Affiliation_Type_Contract,HCO_Affiliation_Type_Referral,TARGET
0,2584,2136011,HCP_2585,56,248,274,409,723,1261,309,...,35,1,2,3,5,9,17,0,0,0
1,1073,7504404,HCP_1074,38,1100,1833,3432,5959,9538,76,...,42,1,2,3,5,8,13,0,0,0
2,1369,5944892,HCP_1370,70,1020,1701,2620,4131,6094,333,...,25,2,3,4,6,7,12,0,0,0
3,1672,3563201,HCP_1673,77,772,1499,2320,2865,5133,223,...,13,2,3,4,4,6,8,0,1,0
4,3637,1512806,HCP_3638,66,698,1364,2074,3945,6197,289,...,40,1,2,4,4,6,7,0,0,0


In [3]:
df_reference.drop(['Unnamed: 0'], axis=1, inplace = True)
df_reference.head()

,NPI_ID,HCP_ID,Age,Number_of_Rx,Rx_last_1_Month,Rx_last_3_Month,Rx_last_6_Month,Rx_last_12_Month,Claims_last_1_Month,Claims_last_3_Month,...,Promotional_medscape_last_6_month,F2F_visit,F2F_visit_last_1_month,F2F_visit_last_3_month,VRC_visit_last_3_month,VRC_visit_last_6_month,VRC_visit_last_12_month,HCO_Affiliation_Type_Contract,HCO_Affiliation_Type_Referral,TARGET
0,2136011,HCP_2585,56,248,274,409,723,1261,309,449,...,35,1,2,3,5,9,17,0,0,0
1,7504404,HCP_1074,38,1100,1833,3432,5959,9538,76,77,...,42,1,2,3,5,8,13,0,0,0
2,5944892,HCP_1370,70,1020,1701,2620,4131,6094,333,582,...,25,2,3,4,6,7,12,0,0,0
3,3563201,HCP_1673,77,772,1499,2320,2865,5133,223,306,...,13,2,3,4,4,6,8,0,1,0
4,1512806,HCP_3638,66,698,1364,2074,3945,6197,289,292,...,40,1,2,4,4,6,7,0,0,0


### Load Inference Data as Current Data

In [4]:
file_path = os.path.join(current_dir, "..", "data","Inference_data", "Inference_data.csv")
file_path = os.path.abspath(file_path)
df_current = pd.read_csv(file_path)
print(df_current.shape)
print('')
df_current.head()

(500, 32)



,Unnamed: 0,NPI_ID,HCP_ID,Age,Number_of_Rx,Rx_last_1_Month,Rx_last_3_Month,Rx_last_6_Month,Rx_last_12_Month,Claims_last_1_Month,...,Promotional_medscape_last_3_month,Promotional_medscape_last_6_month,F2F_visit,F2F_visit_last_1_month,F2F_visit_last_3_month,VRC_visit_last_3_month,VRC_visit_last_6_month,VRC_visit_last_12_month,HCO_Affiliation_Type_Contract,HCO_Affiliation_Type_Referral
0,4637,4982891,HCP_4638,82,133,184,302,330,378,264,...,23,46,2,3,5,5,9,10,0,1
1,3387,2838371,HCP_3388,28,338,425,773,1544,2153,204,...,14,17,1,2,3,4,6,11,1,0
2,2730,4488773,HCP_2731,38,1100,1202,2180,3876,4878,264,...,20,23,2,3,4,5,7,8,0,0
3,593,8623941,HCP_594,84,866,1005,1280,1989,3550,56,...,17,33,2,4,5,4,6,9,1,0
4,2916,8596687,HCP_2917,62,908,1576,2036,2167,4210,103,...,26,46,2,4,7,4,8,10,0,1


In [5]:
df_current.drop(['Unnamed: 0'], axis=1, inplace = True)
df_current.head()

,NPI_ID,HCP_ID,Age,Number_of_Rx,Rx_last_1_Month,Rx_last_3_Month,Rx_last_6_Month,Rx_last_12_Month,Claims_last_1_Month,Claims_last_3_Month,...,Promotional_medscape_last_3_month,Promotional_medscape_last_6_month,F2F_visit,F2F_visit_last_1_month,F2F_visit_last_3_month,VRC_visit_last_3_month,VRC_visit_last_6_month,VRC_visit_last_12_month,HCO_Affiliation_Type_Contract,HCO_Affiliation_Type_Referral
0,4982891,HCP_4638,82,133,184,302,330,378,264,444,...,23,46,2,3,5,5,9,10,0,1
1,2838371,HCP_3388,28,338,425,773,1544,2153,204,328,...,14,17,1,2,3,4,6,11,1,0
2,4488773,HCP_2731,38,1100,1202,2180,3876,4878,264,486,...,20,23,2,3,4,5,7,8,0,0
3,8623941,HCP_594,84,866,1005,1280,1989,3550,56,102,...,17,33,2,4,5,4,6,9,1,0
4,8596687,HCP_2917,62,908,1576,2036,2167,4210,103,179,...,26,46,2,4,7,4,8,10,0,1


### Load the Classifier

In [6]:
# Load the model
file_path = os.path.join(current_dir, "..", "model", "physician_conversion.pkl")
model_path = os.path.abspath(file_path)

with open(model_path, "rb") as f:
    conversion_classifer = pickle.load(f)

In [7]:
drop_id_col_list = ['NPI_ID', 'HCP_ID']
drop_id_col_with_target_list = ['NPI_ID', 'HCP_ID','TARGET']

ref_prediction = conversion_classifer.predict(df_reference.drop(drop_id_col_with_target_list, axis=1))
current_prediction = conversion_classifer.predict(df_current.drop(drop_id_col_list, axis=1))

In [8]:

df_reference['prediction'] = ref_prediction
df_reference['prediction'].value_counts()

0    716
1    184
Name: prediction, dtype: int64

In [9]:
df_current['prediction'] = current_prediction
df_current['TARGET'] = df_current['prediction']

In [10]:
df_current['prediction'].value_counts()

0    381
1    119
Name: prediction, dtype: int64

### Model Monitoring
- Data/Feature Drift
- Model Drift (if possible)

In [12]:
#split for report and test suite
ref_train_data, ref_test_data = train_test_split(df_reference, test_size=0.3, random_state=42)
current_train_data, current_test_data = train_test_split(df_current, test_size=0.3, random_state=42)


#Create Evidently Report
report = Report(
    metrics=[
        DataDriftPreset(),          # Detects if data has drifted
        DataQualityPreset(),        # Checks for missing values, feature distributions
        
    ]
)

# Run the report
report.run(
    reference_data=ref_train_data,
    current_data=current_train_data
)

# Save report as HTML
file_path = os.path.join(current_dir, "..", "reports", "model_monitoring_report.html")
file_path = os.path.abspath(file_path)

report.save_html(file_path)

print("✅ Report generated and saved as 'model_monitoring_report.html'.")


# Step 5: (Optional) Run a Test Suite to automatically PASS/FAIL drift tests
test_suite = TestSuite(
    tests=[
        DataDriftTestPreset(),
        DataQualityTestPreset(),
        
    ]
)

test_suite.run(
    reference_data=ref_test_data,
    current_data=current_test_data
)

# Save test suite results as HTML
file_path = os.path.join(current_dir, "..", "reports", "model_monitoring_tests.html")
file_path = os.path.abspath(file_path)
test_suite.save_html(file_path)

print("✅ Test suite generated and saved as 'model_monitoring_tests.html'.")

e:\Harshit\physician_proj_1\phy_venv\lib\site-packages\scipy\stats\_stats_py.py:8064: RuntimeWarning:

divide by zero encountered in divide

e:\Harshit\physician_proj_1\phy_venv\lib\site-packages\scipy\stats\_stats_py.py:8064: RuntimeWarning:

divide by zero encountered in divide



✅ Report generated and saved as 'model_monitoring_report.html'.


e:\Harshit\physician_proj_1\phy_venv\lib\site-packages\scipy\stats\_stats_py.py:8064: RuntimeWarning:

divide by zero encountered in divide

e:\Harshit\physician_proj_1\phy_venv\lib\site-packages\scipy\stats\_stats_py.py:8064: RuntimeWarning:

divide by zero encountered in divide



✅ Test suite generated and saved as 'model_monitoring_tests.html'.
